# FarmTech Solutions - Fase 6: treinamento, validação e teste

**Integrante(s): PREENCHER NOME COMPLETO E RM.** Renomeie este arquivo para `NomeCompleto_rmXXXXX_pbl_fase6.ipynb` antes de entregar.

Este notebook implementa as Entregas 1 e 2: YOLOv5n ajustado em 30/60 épocas, YOLO COCO sem ajuste e CNN do zero. As células novas ainda precisam ser executadas no Colab; nenhuma saída de treinamento foi inventada. O arquivo `Fase6_FarmTech_Resultados_Verificados.ipynb` acompanha a análise executada dos resultados originais, com as correções.

Dados: Open Images, 40 Bottle e 40 Coffee cup, divisão por classe 32/4/4. As anotações atuais são públicas; ainda falta comprovar a etapa Make Sense AI. Fotografias próprias não são uma exigência do texto fornecido.

Protocolo: semente 42, YOLO 416 px/batch 8/CPU, CNN 128 px. A classificação por imagem e a detecção por caixas são tarefas distintas. Tempos e números podem diferir em novas execuções.


## 1. Google Drive e acesso ao pacote
Crie a pasta `FarmTech_Fase6` no Drive e envie o ZIP corrigido. O notebook extrai os arquivos de trabalho e salva novos resultados separadamente.

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
# Upload the corrected ZIP to this location in Google Drive before running.
ZIP_DRIVE = Path('/content/drive/MyDrive/FarmTech_Fase6/Fase6_FarmTech_Corrigido.zip')
DESTINO = Path('/content/FarmTech_Fase6')
from google.colab import drive
drive.mount('/content/drive')
assert ZIP_DRIVE.is_file(), f'Put the supplied ZIP here or change ZIP_DRIVE: {ZIP_DRIVE}'
import zipfile
DESTINO.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(ZIP_DRIVE) as archive:
    for item in archive.infolist():
        target=(DESTINO/item.filename).resolve()
        assert target.is_relative_to(DESTINO.resolve()), 'Unsafe archive path'
    archive.extractall(DESTINO)
PROJECT=DESTINO/'Fase6_FarmTech_Corrigido'
assert (PROJECT/'train_cnn.py').is_file()
os.chdir(PROJECT)
os.environ['FARMTECH_PROJECT']=str(PROJECT)
print('Project:',PROJECT)


## 2. Dependências e versão do YOLO
Usa-se o código oficial YOLOv5 v7.0. A configuração de carregamento do PyTorch é restrita ao ambiente do projeto para checkpoints legados de origem conhecida. Fontes: [YOLOv5 v7.0](https://github.com/ultralytics/yolov5/tree/v7.0), [serialização PyTorch](https://docs.pytorch.org/docs/2.6/notes/serialization.html).

In [ ]:
# Run before importing NumPy/Pandas/Pillow. If already imported, restart the session.
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements.txt')],check=True)
YOLO=PROJECT/'yolov5'
if not YOLO.exists():
    subprocess.run(['git','clone','--depth','1','--branch','v7.0','https://github.com/ultralytics/yolov5.git',str(YOLO)],check=True)
os.environ['YOLO_REPO']=str(YOLO)
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD']='1'
os.environ['OMP_NUM_THREADS']='3';os.environ['MKL_NUM_THREADS']='3'
os.environ['MPLBACKEND']='Agg'
import torch, numpy as np, pandas as pd, yaml
torch.set_num_threads(3)
print('Torch',torch.__version__,'NumPy',np.__version__,'YOLO',subprocess.check_output(['git','-C',str(YOLO),'rev-parse','HEAD'],text=True).strip())
# CPU makes timing comparisons consistent with supplied historical results.
DEVICE='cpu'


## 3. Partições e integridade
Não há redivisão aleatória: os mesmos conjuntos originais são preservados. As verificações avaliam quantidade por classe, rótulos, limites das caixas, integridade de imagens e duplicatas exatas. Semelhança visual e qualidade das caixas devem ser revisadas manualmente.

In [ ]:
# New results are separate from the supplied historical runs.
NOVOS=PROJECT/'execucoes_novas'/'openimages'
NOVOS.mkdir(parents=True,exist_ok=True)
if not (NOVOS/'dataset').exists():
    shutil.copytree(PROJECT/'openimages'/'dataset',NOVOS/'dataset')
for name in ['selected_images.json','attribution.csv']:
    shutil.copy2(PROJECT/'openimages'/name,NOVOS/name)
for name in ['logs','runs','test_runs','figures','cnn']:
    (NOVOS/name).mkdir(exist_ok=True)
for p in (NOVOS/'dataset'/'labels').glob('*.cache'): p.unlink()
os.environ['FARMTECH_DATA']=str(NOVOS)
data={'path':str(NOVOS/'dataset'),'train':'images/train','val':'images/val','test':'images/test','nc':2,'names':['bottle','coffee_cup']}
(NOVOS/'dataset'/'dataset.yaml').write_text(yaml.safe_dump(data,sort_keys=False))
sys.path.insert(0,str(PROJECT))
import importlib, project_config
importlib.reload(project_config)
from validate_dataset import validate
display(pd.DataFrame(validate(NOVOS)).T)


## 4. Execução e logs
Cada etapa salva sua saída integral e interrompe em caso de falha.

In [ ]:
# Capture complete output and fail clearly if any stage exits with an error.
def executar(args, log_path, cwd=PROJECT):
    log_path=Path(log_path);log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen([sys.executable,*map(str,args)],cwd=str(cwd),env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace')
        for line in process.stdout:
            log.write(line); print(line,end='')
        code=process.wait()
    if code: raise RuntimeError(f'Execution failed ({code}); see {log_path}')

def executar_script(name):
    executar([PROJECT/name],NOVOS/'logs'/f'{Path(name).stem}.log')


## 5. YOLO adaptável - dois treinamentos independentes
Os dois treinos usam os mesmos pesos iniciais COCO; muda a duração. O limite de épocas também altera a programação de learning rate do YOLO, portanto o experimento compara o orçamento/configuração de duração, sem afirmar que todos os updates são idênticos até a época 30.

In [ ]:
# Two independent fine-tuning experiments; both start from original COCO weights.
for epochs in (30,60):
    executar([YOLO/'train.py','--img','416','--batch-size','8','--epochs',str(epochs),
              '--data',NOVOS/'dataset/dataset.yaml','--weights','yolov5n.pt','--workers','2',
              '--project',NOVOS/'runs','--name',f'yolo_n_{epochs}ep','--exist-ok',
              '--seed','42','--patience','0','--device',DEVICE],NOVOS/'logs'/f'yolo{epochs}.log',YOLO)


## 6. Validação e escolha do modelo
A escolha usa exclusivamente validação. O checkpoint `best.pt` segue a função fitness do YOLOv5. Loss da última época e métricas do melhor checkpoint representam momentos diferentes e são identificadas separadamente.

In [ ]:
# Compare models only on validation; test is not used for choosing a model.
validacao=[]
for epochs in (30,60):
    history=pd.read_csv(NOVOS/f'runs/yolo_n_{epochs}ep/results.csv')
    history.columns=history.columns.str.strip()
    # YOLOv5 fitness weights AP50 (0.1) and AP50-95 (0.9).
    fitness=.1*history['metrics/mAP_0.5']+.9*history['metrics/mAP_0.5:0.95']
    best=history.loc[fitness.idxmax()]
    validacao.append({'epochs':epochs,'best_epoch':int(best['epoch'])+1,
        'mAP50':best['metrics/mAP_0.5'],'mAP50_95':best['metrics/mAP_0.5:0.95'],
        'fitness':fitness.max(),'last_train_box_loss':history.iloc[-1]['train/box_loss'],
        'last_val_box_loss':history.iloc[-1]['val/box_loss']})
df_val=pd.DataFrame(validacao);display(df_val)
ESCOLHIDO=int(df_val.sort_values('fitness',ascending=False).iloc[0]['epochs'])
print('Model chosen from validation:',ESCOLHIDO)


## 7. Teste e YOLO tradicional
O código compara as três variantes detectoras nas mesmas oito imagens. COCO bottle=39 e cup=41 são mapeados para as duas classes; cup/coffee cup é uma aproximação. F1 macro é a média dos F1 individuais. mAP usa a mesma implementação interpolada de AP50 para os três detectores.

In [ ]:
# Evaluate both durations after the selection decision is fixed.
for epochs in (30,60):
    executar([YOLO/'val.py','--data',NOVOS/'dataset/dataset.yaml',
        '--weights',NOVOS/f'runs/yolo_n_{epochs}ep/weights/best.pt','--task','test',
        '--img','416','--batch-size','8','--device',DEVICE,
        '--project',NOVOS/'test_runs','--name',f'custom{epochs}','--exist-ok'],NOVOS/'logs'/f'test{epochs}.log',YOLO)
executar_script('compare_detectors.py')
det=json.loads((NOVOS/'detector_comparison.json').read_text())
display(pd.DataFrame([{k:v for k,v in r.items() if k!='per_class'} for r in det]))


## 8. CNN treinada do zero
Quatro blocos convolucionais (16/32/64/128), pooling, dropout 0,4 e duas saídas. Pesos aleatórios, Adam, até 50 épocas, parada após dez épocas sem melhoria; desempate de acurácia pela menor loss de validação. Imagens com mais de uma classe são rejeitadas para não inferir o rótulo a partir da primeira caixa.

In [ ]:
# Full architecture, data augmentation and training loop are in the displayed source.
from IPython.display import Code, display
display(Code(filename=str(PROJECT/'train_cnn.py'),language='python'))
executar_script('train_cnn.py')
executar_script('measure_cnn.py')
cnn=json.loads((NOVOS/'cnn/metrics.json').read_text())
print('Selected epoch:',cnn['best_epoch'],'test accuracy:',cnn['test_accuracy'])
display(pd.DataFrame(cnn['confusion_matrix'],index=['real bottle','real coffee_cup'],columns=['pred bottle','pred coffee_cup']))


## 9. Curvas e imagens processadas
Turquesa indica referências; laranja indica predições. A montagem apresenta o modelo E60, independentemente do escolhido, para comparar a evidência visual histórica.

In [ ]:
executar_script('plot_metrics.py')
executar_script('visualize_test.py')
from IPython.display import Image as DisplayImage
for name in ['curvas_yolo_validacao.png','curvas_cnn.png','predicoes_teste_yolo60.png']:
    display(DisplayImage(filename=str(NOVOS/'figures'/name)))


## 10. Comparação final da execução
Tabela com métricas, treino e latência. Campos de tarefas distintas ficam separados. O resumo é salvo junto às versões do ambiente.


In [ ]:
# Consolidate measured costs and metrics from this run, without mixing tasks.
import re
def tempo_yolo(ep):
    log=(NOVOS/'logs'/f'yolo{ep}.log').read_text(errors='replace')
    found=re.search(rf'{ep} epochs completed in ([0-9.]+) hours',log)
    return float(found.group(1))*3600 if found else None

comparacao=[]
for row in det:
    match=re.search(r'(30|60) ep',row['model'])
    ep=int(match.group(1)) if match else None
    comparacao.append({'modelo':row['model'],'tarefa':'detecção',
      'precision_macro':row['macro_precision_conf025'],'recall_macro':row['macro_recall_conf025'],
      'F1_macro':row['macro_F1_conf025'],'mAP50':row['mAP50_101point'],
      'acuracia_imagem':None,'treino_s':tempo_yolo(ep) if ep else 0,
      'latencia_e2e_ms':row['latency_ms_mean'],'entrada_px':416})
lat=json.loads((NOVOS/'cnn/latency_e2e.json').read_text())
comparacao.append({'modelo':'CNN do zero','tarefa':'classificação',
  'precision_macro':cnn['precision_macro'],'recall_macro':cnn['recall_macro'],
  'F1_macro':cnn['f1_macro'],'mAP50':None,'acuracia_imagem':cnn['test_accuracy'],
  'treino_s':cnn['train_seconds'],'latencia_e2e_ms':lat['mean_ms_image'],'entrada_px':128})
display(pd.DataFrame(comparacao).round(4))
baseline=det[0]
ajustado=next(r for r in det if f'{ESCOLHIDO} ep' in r['model'])
texto=(f'Modelo ajustado escolhido por validação: {ESCOLHIDO} épocas. '
 f'No teste, mAP50 ajustado={ajustado["mAP50_101point"]:.3f} e COCO={baseline["mAP50_101point"]:.3f}; '
 f'F1 macro ajustado={ajustado["macro_F1_conf025"]:.3f} e COCO={baseline["macro_F1_conf025"]:.3f}. '
 f'CNN: acurácia={cnn["test_accuracy"]:.3f}, melhor época={cnn["best_epoch"]}. '
 'Acurácia de classificação não equivale a mAP de detecção. Teste de apenas oito imagens; '
 'latências em CPU, com resoluções e saídas distintas. Não selecionar hiperparâmetros pelo teste.')
print(texto)
(NOVOS.parent/'comparacao_final.json').write_text(json.dumps(comparacao,ensure_ascii=False,indent=2))
(NOVOS.parent/'conclusao_nova_execucao.md').write_text('# Conclusão da nova execução\n\n'+texto,encoding='utf-8')


## 11. Discussão crítica e conclusões

Nos resultados originais, 60 épocas melhoraram mAP em relação a 30. O baseline COCO obteve mAP50 0,676 e F1 macro corrigido 0,564; E60 obteve 0,657 e 0,551. Não houve superioridade absoluta do ajuste. A CNN acertou 6/8 imagens (75%); sua tarefa não localiza objetos.

Após executar, compare as tabelas novas e atualize esta conclusão: facilidade de integração, precisão, tempo de treino e latência. Extraia duração dos logs e use latência ponta a ponta da CNN; não confunda com model-only. Registre hardware. YOLO usa 416 px e CNN 128 px, impedindo concluir superioridade arquitetural apenas pelos tempos.

Limitações: só oito imagens de teste; desbalanceamento de instâncias (19 garrafas e 5 xícaras); origens públicas; taxonomias distintas; possibilidade de contextos semelhantes e sobreajuste. Não usar o teste para ajustar hiperparâmetros. A origem no split validation de Open Images e pesos COCO não garantem ausência de exposição a cenas similares no pré-treinamento.

As entregas Ir Além são opcionais e não foram implementadas neste pacote.


## 12. Persistência no Drive
Execute e salve o próprio notebook com saídas. Não há publicação automática no GitHub ou YouTube.

In [ ]:
# Save experiment files and installed versions in Drive, then save this notebook there too.
VERSOES=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(NOVOS.parent/'requirements_executados.txt').write_text(VERSOES)
(NOVOS.parent/'ambiente.json').write_text(json.dumps({'torch':torch.__version__,'device':DEVICE,
 'yolov5_commit':subprocess.check_output(['git','-C',str(YOLO),'rev-parse','HEAD'],text=True).strip(),
 'selected_epochs_validation':ESCOLHIDO},indent=2))
saida_drive=ZIP_DRIVE.parent/'Resultados_Fase6'
shutil.copytree(NOVOS.parent,saida_drive,dirs_exist_ok=True)
print('Results saved in:',saida_drive)
print('Before submission: save the notebook with its executed cells, add name/RM, and update conclusions from the new results.')
